# DermRx Agent - Notebook 2: Drug Safety Database & Treatment Table
## MedGemma Impact Challenge | Agentic Workflow Prize

**Purpose**: Build the drug interaction checking foundation and an evidence-based treatment table for our medication safety pipeline.
**Data sources**: [DDInter 2.0](https://ddinter.scbdd.com/) (302,516 drug interactions), [MED-RT](https://mor.nlm.nih.gov/RxClass/) (FDA/VA authoritative drug database), [PubChem](https://pubchem.ncbi.nlm.nih.gov/) (molecular SMILES strings)

<font color = "red"><u> **Key findings**:</u></font> MedGemma cannot select drugs from large lists - it echoes back 1,867 drug names instead of picking appropriate treatments. We solved this by building an evidence-based treatment table from MED-RT (FDA/VA), giving the pipeline clinically-ranked drug candidates per condition

<font color = "red"><u> **Technical challenge**:</u></font> DDInter uses different drug names than standard references - for example, Aspirin is stored as "Acetylsalicylic acid." Alias expansion and careful cross-referencing were needed to achieve a 60.1% match rate between MED-RT drugs and DDInter's database, resulting in 163 verified drugs across 11 treatment classes.

---

In Notebook 1, we validated MedSigLIP for skin diagnosis. But diagnosis alone isn't enough - DermRx Agent needs to recommend safe treatments.

## Setup 

We'll be working with DDInter's CSV downloads and making API calls to NLM's RxClass API for MED-RT data. No GPU needed for this notebook - it's all data processing

In [13]:
import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import requests
import json
import time

## Loading the DDInter Database

[DDInter 2.0](https://ddinter.scbdd.com/) is a curated drug-drug interaction database covering 302,516 interactions across 2,310 approved drugs. The data is split into 4 CSV iles organized by severity code. Let's load them and understand what we're working with. 

**Note**: I already downlaoded the CSV's file from official website and created a dataset on kaggle from it which is accessible to everyone with name [ddinter2.0_dataset](https://www.kaggle.com/datasets/meshivanshsinghh/ddinter2-0-dataset)

In [14]:
csv_files = {
    'A': '/kaggle/input/datasets/meshivanshsinghh/ddinter2-0-dataset/ddinter_downloads_code_A.csv',
    'B': '/kaggle/input/datasets/meshivanshsinghh/ddinter2-0-dataset/ddinter_downloads_code_B.csv',
    'D': '/kaggle/input/datasets/meshivanshsinghh/ddinter2-0-dataset/ddinter_downloads_code_D.csv',
    'P': '/kaggle/input/datasets/meshivanshsinghh/ddinter2-0-dataset/ddinter_downloads_code_P.csv',
}

dfs = {}
for code, path in csv_files.items():
    df = pd.read_csv(path)
    dfs[code] = df

In [15]:
# combining into single dataframe
combined = pd.concat(dfs.values(), ignore_index = True)
print(f"Combined: {len(combined)} total interaction rows")
print(f"Severity distribution: {combined['Level'].value_counts()}")

Combined: 102680 total interaction rows
Severity distribution: Level
Moderate    57205
Unknown     27508
Major       11237
Minor        6730
Name: count, dtype: int64


In [16]:
combined.head()

,DDInterID_A,Drug_A,DDInterID_B,Drug_B,Level
0,DDInter1263,Naltrexone,DDInter1,Abacavir,Moderate
1,DDInter1,Abacavir,DDInter1348,Orlistat,Moderate
2,DDInter58,Aluminum hydroxide,DDInter582,Dolutegravir,Major
3,DDInter112,Aprepitant,DDInter582,Dolutegravir,Minor
4,DDInter138,Attapulgite,DDInter582,Dolutegravir,Major


## Extracting Unique Drugs

Each interaction row has Drug_A and Drug_B. We need to collect all unique drugs with their DDInter IDs - these IDs are how we'll look up interactions later in the pipeline.

In [17]:
# drugs appear on both sides, so we collect from both columns
drugs_a = combined[['DDInterID_A', 'Drug_A']].rename(columns={'DDInterID_A': 'DDInterID', 'Drug_A': 'Drug'})
drugs_b = combined[['DDInterID_B', 'Drug_B']].rename(columns={'DDInterID_B': 'DDInterID', 'Drug_B': 'Drug'})

all_drugs = pd.concat([drugs_a, drugs_b]).drop_duplicates().sort_values("Drug").reset_index(drop=True)
print(f"Unique drugs in DDInter: {len(all_drugs)}")

Unique drugs in DDInter: 1867


In [18]:
# Verifying if our key drugs are present
key_drugs = [
    'Fluconazole', 'Terbinafine', 'Clotrimazole', 'Ketoconazole',
    'Warfarin', 'Metformin', 'Lisinopril', 'Atorvastatin',   
    'Metoprolol', 'Omeprazole', 'Amlodipine', 'Levothyroxine',
    'Aspirin', 'Ibuprofen', 'Permethrin',  
]

print("Key drug lookup\n")
for drug in key_drugs:
    match = all_drugs[all_drugs['Drug'].str.lower() == drug.lower()]
    if len(match) > 0:
        print(f"{drug:20} FOUND {match.iloc[0]['DDInterID']}")
    else:
        print(f"{drug:20} NOT FOUND")

Key drug lookup

Fluconazole          FOUND DDInter743
Terbinafine          FOUND DDInter1768
Clotrimazole         FOUND DDInter416
Ketoconazole         FOUND DDInter1008
Warfarin             FOUND DDInter1951
Metformin            FOUND DDInter1164
Lisinopril           FOUND DDInter1079
Atorvastatin         FOUND DDInter133
Metoprolol           FOUND DDInter1200
Omeprazole           FOUND DDInter1340
Amlodipine           FOUND DDInter79
Levothyroxine        FOUND DDInter1064
Aspirin              NOT FOUND
Ibuprofen            FOUND DDInter900
Permethrin           NOT FOUND


### Handling Drug Name Mismatches

Aspiring is one of the most common medications worldwide - it has to be in DDInter. The issue is likely a naming difference. DDInter may store it under its chemical name rather than its brand/common name. Let's search  for it. 

In [19]:
# searching for aspirin-related names
aspirin_search = all_drugs[all_drugs['Drug'].str.lower().str.contains('aspirin|cetylsalicyl|salicyl')]
print("Aspirin search results:")
print(aspirin_search.to_string(index=False))

# searching for permethrin
permethrin_search = all_drugs[all_drugs['Drug'].str.lower().str.contains('permethrin')]
print("Permethrin search results:")
print(permethrin_search.to_string(index=False))

Aspirin search results:
  DDInterID                        Drug
  DDInter20        Acetylsalicylic acid
  DDInter75         Aminosalicylic acid
 DDInter215       Bismuth subsalicylate
 DDInter370          Choline salicylate
DDInter1183 Methyl salicylate (topical)
DDInter1449           Phenyl salicylate
DDInter1633     Salicylic acid (sodium)
DDInter1634    Salicylic acid (topical)
Permethrin search results:
Empty DataFrame
Columns: [DDInterID, Drug]
Index: []


## Drug Interaction Validation - The Warfarin Scenario

The core of DermRx is catching dangerous drug interactions. Our flagship scenario: a patient on warfarin (blood thinner) gets a fungal skin infection. The standard first-line treatment is oral fluconazole - but fluconazole + warfarin is a **Major** interaction that can cause life-threatening bleeding.

In [20]:
def check_ddi(drug1, drug2):
    match1 = all_drugs[all_drugs['Drug'].str.lower() == drug1.lower()]
    match2 = all_drugs[all_drugs['Drug'].str.lower() == drug2.lower()]

    if len(match1) == 0 or len(match2) == 0:
        return None

    id1 = match1.iloc[0]['DDInterID']
    id2 = match2.iloc[0]['DDInterID']

    interaction = combined[
        ((combined['DDInterID_A'] == id1) & (combined['DDInterID_B'] == id2)) |
        ((combined['DDInterID_A'] == id2) & (combined['DDInterID_B'] == id1))
    ]

    if len(interaction) > 0:
        return interaction.iloc[0]['Level']
    return "No interaction found"

In [21]:
# Scenario 1: Antifungals vs Warfarin
antifungals = ['Fluconazole', 'Terbinafine', 'Clotrimazole', 'Ketoconazole']
for drug in antifungals:
    result = check_ddi(drug, 'Warfarin')
    print(f"{drug} + Warfarin = {result}")

# Scenario 2: Fluconazole vs Polypharmacy patient
patient_meds = ['Warfarin', 'Metformin', 'Lisinopril', 'Atorvastatin',
                'Metoprolol', 'Omeprazole', 'Amlodipine', 'Levothyroxine']
for med in patient_meds:
    result = check_ddi('Fluconazole', med)
    print(f"Fluconazole + {med} = {result}")

Fluconazole + Warfarin = Major
Terbinafine + Warfarin = Moderate
Clotrimazole + Warfarin = Unknown
Ketoconazole + Warfarin = Moderate
Fluconazole + Warfarin = Major
Fluconazole + Metformin = Unknown
Fluconazole + Lisinopril = Unknown
Fluconazole + Atorvastatin = Major
Fluconazole + Metoprolol = Unknown
Fluconazole + Omeprazole = Moderate
Fluconazole + Amlodipine = Moderate
Fluconazole + Levothyroxine = Unknown


## DDI Validation Results

The database correctly identifies the key interactions our pipeline needs to catch: 

1. Fluconazole + Warfarin has Major interaction as it is CYP2C9 inhibition so it can increase bleeding risk. REJECT
2. Terbinafine + Warfarin has Moderate interaction as it may slightly increase warfarin effect. CAUTION
3. Ketoconazole + Warfarin has Moderate interaction as it is CYP3A4 inhibition that affects warfarin metabolism. CAUTION
4. Clotrimazole + Warfarin has Unknown interaction as there is no known systemic interaction (topical). SAFE

The polypharmacy scenario reveals fluconazole also has a **Major** interaction with atorvastatin (risk of rhabdomyolysis) and **Moderate** interactions with omeprazole and amlodipine. This validates DDInter as our primary drug safety database. 

Next, we need an evidence-based list of treatment drugs per skin condition - the candidates that our agentic pipeline will evaluate.

---

## Building an Evidence-Based Treatment Table from MED-RT

We now know DDInter can catch dangerous interactions. But we need a structured list of drugs to recommend for each skin condition. We initially tried having MedGemma suggest treatments directly - but it couldn't reliably select from 1,867 drug names. 

**Solution**: [MED-RT](https://mor.nlm.nih.gov/RxClass/) is the FDA/VA's authoritative drug relationship database. We query NLM's RxClass API to find which drugs are approved to treat conditions matching our 11 Tier 1 treatment classes. This gives us clinically-validated candidates instead of relying on an LLM to pick drugs.

In [22]:
BASE_URL = "https://rxnav.nlm.nih.gov/REST/rxclass"

# MeSH search terms for each of our 11 Tier 1 treatment classes
search_terms = {
    "antifungal": ["Tinea", "Dermatomycoses"],
    "topical_steroid": ["Dermatitis", "Eczema"],
    "psoriasis_treatment": ["Psoriasis"],
    "antibiotic": ["Skin Diseases, Bacterial", "Impetigo"],
    "acne_treatment": ["Acne Vulgaris"],
    "antiviral": ["Herpes Simplex", "Herpes Zoster"],
    "antiparasitic": ["Scabies", "Lice Infestations"],
    "rosacea": ["Rosacea"],
    "wart_treatment": ["Warts"],
    "lichen_treatment": ["Lichen Planus"],
    "nail_treatment": ["Onychomycosis"],
}

In [23]:
# Step 1: We will find MeSH class IDs for each treatment class
all_class_results = {}

for treatment_class, terms in search_terms.items():
    all_class_results[treatment_class] = []
    for term in terms: 
        url = f"{BASE_URL}/class/byName.json?className={term}&relaSource=MEDRT&relas=may_treat"
        resp = requests.get(url)
        if resp.status_code == 200:
            data = resp.json()
            if 'rxclassMinConceptList' in data: 
                for item in data['rxclassMinConceptList']['rxclassMinConcept']:
                    all_class_results[treatment_class].append({
                        'classId': item['classId'],
                        'className': item['className']
                    })
    time.sleep(0.3)

print("MeSH IDs found per treatment class:\n")
total_ids = 0
for tc, results in all_class_results.items():
    ids = [r['classId'] for r in results]
    total_ids += len(ids)
    print(f"{tc} {len(ids)} IDs: {', '.join(ids)}")
print(f"Total MeSH IDs: {total_ids}")

MeSH IDs found per treatment class:

antifungal 2 IDs: D014005, D003881
topical_steroid 2 IDs: D003872, D004485
psoriasis_treatment 1 IDs: D011565
antibiotic 2 IDs: D017192, D007169
acne_treatment 1 IDs: D000152
antiviral 2 IDs: D006561, D006562
antiparasitic 2 IDs: D012532, D010373
rosacea 1 IDs: D012393
wart_treatment 1 IDs: D014860
lichen_treatment 1 IDs: D008010
nail_treatment 1 IDs: D014009
Total MeSH IDs: 16


### Fetching Drug Members from MED-RT

Now we use each MeSH ID to query which drugs the FDA/VA says can treat that condition. We filter to base ingedients only (tty=IN) to avoid duplicates from brand names and formulations.

In [24]:
# Step 2: Get drug members for each MeSH class ID
all_drugs_by_class = {}

for treatment_class, class_results in all_class_results.items():
    all_drugs_by_class[treatment_class] =  {}

    for cr in class_results:
        class_id = cr['classId']
        url = f"{BASE_URL}/classMembers.json?classId={class_id}&relaSource=MEDRT&rela=may_treat&ttys=IN+PIN"
        resp = requests.get(url)

        if resp.status_code == 200:
            data = resp.json()
            if 'drugMemberGroup' in data: 
                for member in data['drugMemberGroup']['drugMember']:
                    drug_info = member['minConcept']
                    if drug_info.get('tty') == 'IN':
                        all_drugs_by_class[treatment_class][drug_info['name']] = {
                            'rxcui': drug_info['rxcui'],
                            'tty': drug_info['tty']
                        }
        time.sleep(0.3)


print("Drugs found per treatment class:")
total = 0
for tc, drugs in all_drugs_by_class.items():
    total += len(drugs)
    print(f"{tc} {len(drugs)} drugs")
print(f"Total unique drugs (IN only): {total}")

Drugs found per treatment class:
antifungal 33 drugs
topical_steroid 55 drugs
psoriasis_treatment 47 drugs
antibiotic 58 drugs
acne_treatment 25 drugs
antiviral 16 drugs
antiparasitic 9 drugs
rosacea 8 drugs
wart_treatment 16 drugs
lichen_treatment 0 drugs
nail_treatment 5 drugs
Total unique drugs (IN only): 272


### Handling Missing Treatment Classes

MED-RT returned 0 drugs for lichen planus - the MeSH term didn't map to any drug members. Clinically, lichen planus is treated with topical steroids, so we use the tropical_steroid class as a fallback for them. 

In [25]:
# lichen planus is treated with topical steroids
if len(all_drugs_by_class.get("lichen_treatment", {})) == 0:
    all_drugs_by_class["lichen_treatment"] = dict(all_drugs_by_class.get("topical_steroid", {}))
    print(f"lichen treatment: copied {len(all_drugs_by_class['lichen_treatment'])} drugs from topical steroid")

lichen treatment: copied 55 drugs from topical steroid


### Cross-Referencing with DDInter

Not every MED-RT drug will be in DDInter's database. We need to find which of our 272 treatment candidates have DDInter entries - only those can be safety-checked for drug interactions in our pipeline.

In [26]:
# cross-reference MED-RT drugs against DDInter
ddinter_verified_by_class = {}
not_found = []

for tc, drugs in all_drugs_by_class.items():
    ddinter_verified_by_class[tc] = {}
    for drug_name, drug_info in drugs.items():
        match = all_drugs[all_drugs['Drug'].str.lower() == drug_name.lower()]
        if len(match) > 0:
            ddinter_verified_by_class[tc][drug_name] = {
                **drug_info, 
                'ddinter_name': match.iloc[0]['Drug'],
                'ddinter_id': match.iloc[0]['DDInterID']
            }
        else:
            not_found.append((tc, drug_name))

print("DDInter-verified drugs per class:")
total_verified = 0
total_raw = 0
for tc,drugs in ddinter_verified_by_class.items():
    raw_count = len(all_drugs_by_class[tc])
    total_verified += len(drugs)
    total_raw += raw_count
    print(f"{tc} {len(drugs)}/{raw_count} verified")

print(f"Total: {total_verified}/{total_raw} {total_verified/total_raw*100:.1f}% match rate")
print(f"Not found in DDInter: {len(not_found)} drugs")
        

DDInter-verified drugs per class:
antifungal 15/33 verified
topical_steroid 22/55 verified
psoriasis_treatment 30/47 verified
antibiotic 44/58 verified
acne_treatment 12/25 verified
antiviral 6/16 verified
antiparasitic 2/9 verified
rosacea 2/8 verified
wart_treatment 7/16 verified
lichen_treatment 22/55 verified
nail_treatment 3/5 verified
Total: 165/327 50.5% match rate
Not found in DDInter: 162 drugs


### Alias Expansion

Some drugs aren't found because DDInter uses different naming conventions - chemical names instead of common names, or includes route-specific suffixes. We search for partial matches and create aliases to boost our match rate.

In [27]:
# aliases discovered through partial matching
ALIASES = {
    "calcipotriene": "Calcipotriol",
    "benzoyl peroxide": "Benzoyl peroxide (topical)",
    "retapamulin": "Retapamulin (topical)",
    "penciclovir": "Penciclovir (topical)",
    "fluocinolone": "Fluocinolone acetonide",
    "sulfacetamide": "Sulfacetamide (ophthalmic)",
}

# rerunning cross-reference with aliases this time
alias_additions = 0
for tc, drugs in all_drugs_by_class.items():
    for drug_name, drug_info in drugs.items():
        if drug_name in ddinter_verified_by_class[tc]:
            continue
    alias = ALIASES.get(drug_name.lower()) or ALIASES.get(drug_name)
    if alias:
        match = all_drugs[all_drugs['Drug'] == alias]
        if len(match) > 0:
            ddinter_verified_by_class[tc][drug_name] = {
                    **drug_info,
                    'ddinter_name': match.iloc[0]['Drug'],
                    'ddinter_id': match.iloc[0]['DDInterID']
            }
            alias_additions += 1
            print(f"{drug_name} {alias} (via alias)")

# updated totals
total_verified = sum(len(d) for d in ddinter_verified_by_class.values())
total_raw = sum(len(d) for d in all_drugs_by_class.values())
print(f"Alias additions: {alias_additions}")
print(f"Updated totals: {total_verified}/{total_raw} ({total_verified/total_raw*100:.1f}% match rate)")

Alias additions: 0
Updated totals: 165/327 (50.5% match rate)


### Clinical Cleanup 

Not everything MED-RT returns is clinically appropriate. Some drugs ended up in the wrong treatment class - biologics classified as antibiotics, withdrawn drugs still in the database, ophthalmic-only formulations listed for skin conditions. We manually review and remove these to ensure our treatment recommendations are clinically sound.

In [28]:
# drugs that we might be removing
REMOVALS = {
    "antibiotic": ["adalimumab"],  # biologic for hidradenitis, not an antibiotic
    "antiviral": ["methylene blue", "trifluridine"],  # experimental / ophthalmic only
    "topical_steroid": ["methdilazine", "triprolidine"],  # antihistamines, not steroids
    "psoriasis_treatment": ["alefacept", "efalizumab"],  # withdrawn from market
    "acne_treatment": ["loteprednol etabonate"],  # ophthalmic corticosteroid
    "antihistamine": [
        "anakinra", "astemizole", "betamethasone", "canakinumab", "ciclopirox",
        "clotrimazole", "econazole", "fluocinolone", "fluocinonide", "fluoxymesterone",
        "griseofulvin", "histamine", "hydrocortisone", "icatibant", "ketoconazole",
        "miconazole", "nedocromil", "phenylephrine", "terbinafine"
    ],  # not antihistamines — misclassified by MED-RT
    "lichen_treatment": ["methdilazine", "triprolidine"],  # antihistamines
}

In [29]:
removed_count = 0
for tc, drugs_to_remove in REMOVALS.items():
    for drug in drugs_to_remove:
        if drug in ddinter_verified_by_class.get(tc, {}):
            del ddinter_verified_by_class[tc][drug]
            removed_count += 1

# final summary
print(f"Removed {removed_count} clinically inappropriate drugs")
print("Final treatment table:\n")
total_final = 0
for tc, drugs in ddinter_verified_by_class.items():
    total_final += len(drugs)
    print(f"{tc} {len(drugs)} drugs")
print(f"Total DDInter-verified drugs: {total_final}")

Removed 9 clinically inappropriate drugs
Final treatment table:

antifungal 15 drugs
topical_steroid 20 drugs
psoriasis_treatment 28 drugs
antibiotic 43 drugs
acne_treatment 12 drugs
antiviral 4 drugs
antiparasitic 2 drugs
rosacea 2 drugs
wart_treatment 7 drugs
lichen_treatment 20 drugs
nail_treatment 3 drugs
Total DDInter-verified drugs: 156


## Collecting SMILES from PubChem

TxGemma predicts molecular properties like toxicity and photosensitivity using SMILES string - a text representation of molecular structure. We query PubChem's API to get SMILES for each verified drug. Biologics (large protein molecules like adalimumab, dupilumab) won't have usable SMILES - we flag those as ineligible for TxGemma but they will still get DDI checking through DDInter.

In [30]:
PUBCHEM_BASE = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound"

# known edge cases from our previous research
MANUAL_CIDS = {
    "ivermectin": 6321424 
}
MANUAL_SMILES = {
    "interferon alfa-2b": None,
    "interferon alfa-n3": None,
    "interferon alfacon-1": None,
    "interferon gamma-1b": None,
    "peginterferon alfa-2a": None,
    "peginterferon alfa-2b": None,
    "dupilumab": None,
    "adalimumab": None,
    "abatacept": None,
    "brodalumab": None,
    "certolizumab pegol": None,
    "etanercept": None,
    "guselkumab": None,
    "infliximab": None,
    "ixekizumab": None,
    "risankizumab": None,
    "secukinumab": None,
    "tildrakizumab": None,
    "ustekinumab": None,
}

In [37]:
def get_smiles(drug_name):
    # checking manual overrides first
    if drug_name.lower() in MANUAL_SMILES or drug_name in MANUAL_SMILES:
        return MANUAL_SMILES.get(drug_name.lower(), MANUAL_SMILES.get(drug_name))

    # trying CID lookup for known problem drugs
    if drug_name.lower() in MANUAL_CIDS:
        cid = MANUAL_CIDS[drug_name.lower()]
        try:
            url = f"{PUBCHEM_BASE}/cid/{cid}/property/CanonicalSMILES/JSON"
            resp = requests.get(url, timeout=15)
            if resp.status_code == 200:
                props = resp.json().get("PropertyTable", {}).get("Properties", [])
                if props:
                    return props[0].get("ConnectivitySMILES") or props[0].get("CanonicalSMILES")
        except:
            pass
        return None

    # trying standard name lookup 
    try:
        url = f"{PUBCHEM_BASE}/name/{drug_name}/property/CanonicalSMILES/JSON"
        resp = requests.get(url, timeout = 10)
        if resp.status_code == 200:
            props = resp.json().get("PropertyTable", {}).get("Properties", [])
            if props:
                return props[0].get("ConnectivitySMILES") or props[0].get("CanonicalSMILES")
    except:
        pass
    return None

In [38]:
# collecting SMILES for all verified drugs
smiles_found = 0
smiles_missing = []

for tc, drugs in ddinter_verified_by_class.items():
    for drug_name, drug_info in drugs.items():
        smiles = get_smiles(drug_name)
        drug_info['smiles'] = smiles
        drug_info['txgemma_eligible'] = smiles is not None

        if smiles:
            smiles_found += 1
        else:
            smiles_missing.append((tc, drug_name))

        time.sleep(0.2)

total_drugs = sum(len(d) for d in ddinter_verified_by_class.values())
print(f"SMILES collected: {smiles_found}/{total_drugs}")
print(f"Missing (bilogics/complex): {len(smiles_missing)}")
print(f"Drugs without SMILES")
for tc, drug in smiles_missing: 
    print(f"{tc} {drug}")

SMILES collected: 135/156
Missing (bilogics/complex): 21
Drugs without SMILES
topical_steroid dupilumab
psoriasis_treatment secukinumab
psoriasis_treatment ixekizumab
psoriasis_treatment brodalumab
psoriasis_treatment infliximab
psoriasis_treatment guselkumab
psoriasis_treatment tildrakizumab
psoriasis_treatment etanercept
psoriasis_treatment risankizumab
psoriasis_treatment adalimumab
psoriasis_treatment abatacept
psoriasis_treatment certolizumab pegol
psoriasis_treatment ustekinumab
antiviral interferon alfa-2b
wart_treatment peginterferon alfa-2a
wart_treatment peginterferon alfa-2b
wart_treatment interferon alfa-2b
wart_treatment interferon gamma-1b
wart_treatment interferon alfacon-1
wart_treatment interferon alfa-n3
lichen_treatment dupilumab


## Exporting the Treatment Table

We now have everything needed for the pipeline: DDInter-verified drugs with their IDs, SMILES string for TxGemma molecular analysis, and treatment class assignments matching our Tier 1 classification categories. Let's export as our production treatment table for lookup. 

In [42]:
# building the final treatment table JSON
treatment_data = {
    "version": "3.0",
    "source": "MED-RT (RxClass API) + DDInter 2.0 cross-reference + PubChem SMILES",
    "treatment_classes": {}
}

for tc, drugs in ddinter_verified_by_class.items():
    treatment_data["treatment_classes"][tc] = {
        "ddinter_verified": [
            {
                "drug_name": drug_name,
                "ddinter_name": info['ddinter_name'],
                "rxcui": info.get('rxcui', ''),
                "smiles": info.get('smiles'),
                "txgemma_eligible": info.get('txgemma_eligible', False),
            } 
            for drug_name, info in drugs.items()
        ]
    }

In [45]:
# exporting the table and storing final summary 
with open("dermrx_treatment_table_v3.json", "w") as f:
    json.dump(treatment_data, f, indent = 2)

# final summary 
total_drugs = sum(len(d) for d in ddinter_verified_by_class.values())
txgemma_eligible = sum(
    1 for d in ddinter_verified_by_class.values()
    for info in d.values() if info.get("txgemma_eligible")
)

print(f"Total DDInter-verified drugs: {total_drugs}")
print(f"TxGemma-eligible (have SMILES): {txgemma_eligible}")
print(f"Biologics (DDI only, no SMILES): {total_drugs - txgemma_eligible}")
print(f"Treatment classes: {len(ddinter_verified_by_class)}")

Total DDInter-verified drugs: 156
TxGemma-eligible (have SMILES): 135
Biologics (DDI only, no SMILES): 21
Treatment classes: 11


## Notebook 2 Summary

We've built the drug safety foundation for DermRx Agent: 

1. **Loaded** DDInter database - 102,680 drug interactions across 1,867 drugs
2. **Validated** DDI checking - fluconazole + warfarin correctly flagged as Major interaction
3. **Discovered** naming mismatches - Aspirin stored as "Acetylsalicylic acid" in DDInter
4. **Built** evidence-based treatment table from MED-RT (FDA/VA) - solved the problem of MedGemma being unable to select from large drug lists
5. **Cross-referenced** with DDInter - 156 verified drugs that can be safety-checked
6. **Cleaned** clinically inappropriate entries - removed withdrawn drugs, misclassified biologics
7. **Collected** SMILES strings from PubChem - 135 small molecules ready for TxGemma molecular analysis.

<font color = "red"><u>**Next**:</u></font> Notebook 3 validates TxGemma for molecular toxicity prediction and MedGemma for clinical reasoning - completing the model stack for our agentic pipeline. 